# RNN과 LSTM: 시계열 예측과 셰익스피어 생성기

**30일 딥러닝 완전 정복 · Day 20** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 사인파 시계열 예측으로 RNN과 LSTM의 장기 기억 능력을 비교한다
- 문자 단위 LSTM 언어 모델을 학습하고 temperature를 바꿔 텍스트를 생성한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

## 1. 장기 기억 테스트
입력 시퀀스의 **첫 번째 값**을 마지막에 맞히는 과제입니다. 중간은 모두 잡음입니다. 시퀀스가 길어질수록 바닐라 RNN은 첫 값을 잊어버립니다.

In [ ]:
def make_batch(B, T):
    x = torch.randn(B, T, 1) * 0.5
    y = x[:, 0, 0].clone()                        # 정답 = 첫 값
    return x.to(device), y.to(device)

class SeqReg(nn.Module):
    def __init__(self, cell="lstm", hidden=64):
        super().__init__()
        rnn = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn(1, hidden, batch_first=True); self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)                      # (B, T, H)
        return self.head(out[:, -1]).squeeze(-1)  # 마지막 시점으로 예측

def run(cell, T, steps=1500):
    m = SeqReg(cell).to(device); opt = torch.optim.Adam(m.parameters(), 3e-3)
    for s in range(steps):
        x, y = make_batch(128, T); loss = F.mse_loss(m(x), y)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    with torch.no_grad(): x, y = make_batch(1000, T); return F.mse_loss(m(x), y).item()

print("기준선(항상 0 예측) MSE ≈ 0.25")
for T in [5, 20, 50]:
    print(f"T={T:3d}  " + "  ".join(f"{c}: {run(c, T):.4f}" for c in ["rnn", "gru", "lstm"]))

## 2. 문자 단위 언어 모델: 셰익스피어

In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} chars, vocab {V}"); print(text[:300])

def get_batch(split, T=128, B=64):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - T - 1, (B,))
    x = torch.stack([d[i:i+T] for i in ix]); y = torch.stack([d[i+1:i+T+1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, V, emb=64, hidden=512, layers=2):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.lstm = nn.LSTM(emb, hidden, layers, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden, V)
    def forward(self, idx, state=None):
        out, state = self.lstm(self.emb(idx), state)
        return self.head(out), state

@torch.no_grad()
def generate(model, start="ROMEO:", n=400, temperature=0.8):
    model.eval(); idx = torch.tensor([[stoi[c] for c in start]], device=device); out = list(start); state = None
    logits, state = model(idx, state)
    for _ in range(n):
        probs = F.softmax(logits[0, -1] / temperature, -1)
        nxt = torch.multinomial(probs, 1).item(); out.append(itos[nxt])
        logits, state = model(torch.tensor([[nxt]], device=device), state)
    model.train(); return "".join(out)

model = CharLSTM(V).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params,  initial loss should be ~ln(V) = {math.log(V):.2f}")
for it in range(3001):
    x, y = get_batch("train")
    logits, _ = model(x); loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    if it % 500 == 0:
        with torch.no_grad():
            xv, yv = get_batch("val"); vl = F.cross_entropy(model(xv)[0].reshape(-1, V), yv.reshape(-1)).item()
        print(f"\n=== iter {it}  train {loss.item():.3f}  val {vl:.3f} ===")
        print(generate(model, n=200))

## 3. 온도 비교

In [ ]:
for t in [0.3, 0.8, 1.5]:
    print(f"\n----- temperature {t} -----"); print(generate(model, "JULIET:", 300, t))

## 과제
- 세 온도의 결과를 학습 노트에 붙이고 특징을 비교하세요.
- (도전) `text`를 여러분이 고른 한국어 텍스트로 바꿔 학습해 보세요 (저작권 유의).